In [1]:
import gc
import re
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchaudio

torch.set_grad_enabled(False)

REPO_ROOT = Path("/home/amin/Research/Representation/Movie/Model Repos/cav-mae-sync")
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import models

print("Imports loaded successfully")


/home/amin/miniconda3/envs/cav-mae-sync/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded successfully


In [2]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
REPO_ROOT = Path("/home/amin/Research/Representation/Movie/Model Repos/cav-mae-sync")
BASE_DIR  = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
DEFAULT_MODEL_PATH  = REPO_ROOT / "pretrained_models" / "cav_mae_sync.pth"
DEFAULT_OUTPUTS_ROOT = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync")

# AudioSet log-mel stats (valid only with the Kaldi/HTK fbank pipeline below)
AUDIO_MEAN = -5.081
AUDIO_STD  = 4.4849

TOTAL_FRAMES      = 16
BATCH_SIZE        = 4        # reduce to 2 if 12 GB VRAM is tight at 10 s segments
FORCE_CPU         = True    # set True only for debugging; use GPU for real runs
SEGMENT_DURATIONS = [2.0]


# Mel time frames per segment duration.
# MUST BE A MULTIPLE OF 16 for the MAE patch embedding to work correctly.
# 1 s  ->  96 frames   (closest to 100)
# 2 s  ->  192 frames  (closest to 200)
# 5 s  ->  496 frames  (closest to 500)
# 10 s ->  992 frames  (closest to 1000)
TARGET_LENGTHS = {
    1.0:  96,
    2.0:  192,
    5.0:  496,
    10.0: 992,
}

# ---------------------------------------------------------------------------
# Derived
# ---------------------------------------------------------------------------
USE_CUDA = (not FORCE_CPU) and torch.cuda.is_available()
DEVICE   = torch.device("cuda" if USE_CUDA else "cpu")

print(f"Base data directory : {BASE_DIR}")
print(f"Model checkpoint    : {DEFAULT_MODEL_PATH}")
print(f"Output root         : {DEFAULT_OUTPUTS_ROOT}")
print(f"Device              : {DEVICE}")
print(f"Segment durations   : {SEGMENT_DURATIONS}")
print(f"Target lengths      : {TARGET_LENGTHS}")
print(f"Batch size          : {BATCH_SIZE}")


Base data directory : /home/amin/Research/Representation/Movie/data/Setareh/Data
Model checkpoint    : /home/amin/Research/Representation/Movie/Model Repos/cav-mae-sync/pretrained_models/cav_mae_sync.pth
Output root         : /home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync
Device              : cpu
Segment durations   : [2.0]
Target lengths      : {1.0: 96, 2.0: 192, 5.0: 496, 10.0: 992}
Batch size          : 4


In [3]:
# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------

def extract_part_number(p: Path):
    """Return the integer part number embedded in a filename, or None."""
    m = re.search(r'part[_-]?(\d+)', p.stem, re.IGNORECASE)
    return int(m.group(1)) if m else None


def gather_pairs_for_segment(seg_sec: float):
    """
    Walk the 18 video runs and collect paired (video_path, audio_path) tuples
    for the given segment duration.

    Directory layout expected:
        BASE_DIR/Video{n}/Video{n}_chunks_{k}s/*.mp4
        BASE_DIR/Audio{n}/Audio{n}_chunks_{k}s/*.wav  (or *.m4a fallback)
    """
    all_videos, all_audios = [], []
    k = int(seg_sec)

    for run_idx in range(1, 19):
        vid_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks_{k}s"
        aud_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks_{k}s"

        if not vid_dir.exists() or not aud_dir.exists():
            continue

        vids = sorted(vid_dir.glob("*.mp4"),  key=lambda p: extract_part_number(p) or 0)
        auds = sorted(aud_dir.glob("*.wav"),   key=lambda p: extract_part_number(p) or 0)
        if not auds:
            auds = sorted(aud_dir.glob("*.m4a"), key=lambda p: extract_part_number(p) or 0)

        vid_map = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_map = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}
        common  = sorted(vid_map.keys() & aud_map.keys())

        if not common:
            print(f"  Video {run_idx}: no common part numbers found "
                  f"(vid_dir={vid_dir.exists()}, aud_dir={aud_dir.exists()}, "
                  f"n_vids={len(vid_map)}, n_auds={len(aud_map)})")
            continue

        for part in common:
            all_videos.append(vid_map[part])
            all_audios.append(aud_map[part])

        status = "OK" if len(vid_map) == len(aud_map) else "MISMATCH"
        print(f"  Video {run_idx}: {len(common)} paired chunks [{status}: "
              f"{len(vid_map)} vids / {len(aud_map)} auds]")

    paired = list(zip(all_videos, all_audios))
    return paired, all_videos, all_audios


def infer_register_tokens(state_dict: dict) -> int:
    """
    The checkpoint stores register_tokens with shape (num_register_tokens*2, 1, D)
    because audio and video registers are concatenated.  Dividing by 2 recovers
    the per-modality count expected by the model constructor.
    """
    for key in ("register_tokens", "module.register_tokens"):
        if key in state_dict:
            n = state_dict[key].shape[0] // 2
            assert n > 0, f"Unexpected register_tokens shape {state_dict[key].shape}"
            return n
    print("  WARNING: register_tokens not found in checkpoint, defaulting to 4")
    return 4


def _resize_pos_embed(state_dict: dict, model_state_dict: dict, key: str):
    """
    Linearly interpolate a positional embedding from the checkpoint to the
    target length required by the model (happens when segment duration differs
    from the training default of ~4 s / 400 mel frames).
    """
    if key not in state_dict or key not in model_state_dict:
        return
    src, dst = state_dict[key], model_state_dict[key]
    if src.shape == dst.shape:
        return
    if src.ndim != 3 or dst.ndim != 3:
        return
    # (1, T_src, D) -> (1, D, T_src) -> interp -> (1, D, T_dst) -> (1, T_dst, D)
    src_t = src.permute(0, 2, 1).float()
    src_t = F.interpolate(src_t, size=dst.shape[1], mode="linear", align_corners=False)
    state_dict[key] = src_t.permute(0, 2, 1).to(src.dtype)
    print(f"  Resized {key}: {tuple(src.shape)} -> {tuple(state_dict[key].shape)}")


In [4]:
# ---------------------------------------------------------------------------
# AUDIO / VIDEO LOADING
# ---------------------------------------------------------------------------

def load_audio_fbank(wav_path: Path, target_length: int) -> torch.Tensor:
    """
    Compute a log-mel spectrogram using torchaudio.compliance.kaldi.fbank,
    which matches the Kaldi/HTK pipeline used to derive AUDIO_MEAN/AUDIO_STD.

    Returns: (target_length, 128) float32 tensor, normalised.
    """
    waveform, sr = torchaudio.load(str(wav_path))

    # Stereo -> mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample to 16 kHz
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    # Kaldi fbank: 128 mel bins, 25 ms window, 10 ms hop, HTK scale
    fbank = torchaudio.compliance.kaldi.fbank(
        waveform,
        htk_compat=True,
        sample_frequency=16000,
        use_energy=False,
        window_type="hanning",
        num_mel_bins=128,
        dither=0.0,
        frame_shift=10,
    )
    # fbank shape: (T, 128)

    # Normalise with AudioSet statistics
    fbank = (fbank - AUDIO_MEAN) / (AUDIO_STD * 2)

    # Pad or truncate to target_length along the time axis
    T = fbank.shape[0]
    if T < target_length:
        fbank = F.pad(fbank, (0, 0, 0, target_length - T))
    else:
        fbank = fbank[:target_length, :]

    return fbank  # (target_length, 128)


def load_video_frames(video_path: Path, num_frames: int = 16) -> torch.Tensor:
    """
    Uniformly sample `num_frames` frames from an MP4 file.
    Returns: (num_frames, 3, 224, 224) float32 tensor, values in [0, 1].
    Attempts torchvision first, falls back to OpenCV, then black frames.
    """
    # --- torchvision ---
    try:
        from torchvision.io import read_video
        video_data, _, _ = read_video(str(video_path), output_format="TCHW", pts_unit="sec")
        if video_data.shape[0] == 0:
            raise ValueError("Empty video")
        indices = np.linspace(0, video_data.shape[0] - 1, num_frames, dtype=int)
        frames = video_data[indices].float() / 255.0        # (N, 3, H, W)
        if frames.shape[-2:] != (224, 224):
            frames = F.interpolate(frames, size=(224, 224), mode="bilinear", align_corners=False)
        return frames  # (num_frames, 3, 224, 224)
    except Exception as e_tv:
        pass

    # --- OpenCV fallback ---
    try:
        import cv2
        cap = cv2.VideoCapture(str(video_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0:
            raise ValueError("Empty video")
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (224, 224))
                frames.append(torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0)
            else:
                frames.append(torch.zeros(3, 224, 224))
        cap.release()
        while len(frames) < num_frames:
            frames.append(torch.zeros(3, 224, 224))
        return torch.stack(frames[:num_frames])
    except Exception as e_cv:
        print(f"  WARNING: both video decoders failed for {video_path}; using black frames")
        return torch.zeros(num_frames, 3, 224, 224)


In [5]:
# ---------------------------------------------------------------------------
# MODEL LOADING
# ---------------------------------------------------------------------------

def load_model(target_length: int):
    """
    Load CAV-MAE-Sync from checkpoint, resizing positional embeddings if needed.
    Returns the model in eval mode on DEVICE.
    """
    state = torch.load(DEFAULT_MODEL_PATH, map_location="cpu")
    num_reg = infer_register_tokens(state)

    # --- NEW: Strip the 'module.' prefix from the checkpoint keys ---
    clean_state = {}
    for k, v in state.items():
        # Remove 'module.' if it exists at the start of the key
        new_key = k[7:] if k.startswith("module.") else k
        clean_state[new_key] = v
    # ----------------------------------------------------------------

    model = models.CAVMAESync(
        audio_length=target_length,
        modality_specific_depth=11,
        num_register_tokens=num_reg,
        cls_token=True,
        total_frame=TOTAL_FRAMES,
    )

    # Resize positional embeddings using the cleaned state
    for key in ("pos_embed_a", "pos_embed_v", "decoder_pos_embed_a", "decoder_pos_embed_v"):
        _resize_pos_embed(clean_state, model.state_dict(), key)

    # Load the cleaned state dictionary
    missing, unexpected = model.load_state_dict(clean_state, strict=False)
    if missing:
        print(f"  Missing keys   : {missing[:5]}{'...' if len(missing) > 5 else ''}")
    if unexpected:
        print(f"  Unexpected keys: {unexpected[:5]}{'...' if len(unexpected) > 5 else ''}")

    model = model.to(DEVICE)
    model.eval()
    return model

In [6]:
# ---------------------------------------------------------------------------
# EMBEDDING EXTRACTION
# ---------------------------------------------------------------------------

def extract_embeddings_for_segment_duration(seg_sec: float):
    """
    For each paired (video, audio) segment of duration `seg_sec`:
      1. Load mel-spectrogram  -> (target_length, 128)
      2. Load video frames     -> (TOTAL_FRAMES, 3, 224, 224)
      3. Run forward_feat      -> tokens_a (B, T_a, D), tokens_v (B, T_v, D)
      4. Extract CLS token (index 0) from each modality
      5. Concatenate -> (D_a + D_v,) per segment

    The CLS token is the correct clip-level summary: the model is trained with
    cls_token=True and the CLS token aggregates the entire sequence via
    self-attention.  Mean-pooling all patch tokens is less accurate.
    """
    print(f"\n{'='*70}")
    print(f"Processing {int(seg_sec)}s segments")
    print(f"{'='*70}")

    paired, _, _ = gather_pairs_for_segment(seg_sec)
    if not paired:
        print(f"No paired segments found for {int(seg_sec)}s")
        return None, None

    print(f"Found {len(paired)} paired segments")

    target_length = TARGET_LENGTHS[seg_sec]
    print(f"Loading model (target_length={target_length})...")
    model = load_model(target_length)
    print(f"Model loaded on {DEVICE}")

    output_path = DEFAULT_OUTPUTS_ROOT / f"{int(seg_sec)}s"
    output_path.mkdir(parents=True, exist_ok=True)

    all_embeddings = []

    with torch.inference_mode(), torch.amp.autocast("cuda", enabled=USE_CUDA):
        for batch_start in range(0, len(paired), BATCH_SIZE):
            batch_pairs = paired[batch_start : batch_start + BATCH_SIZE]
            n = len(batch_pairs)

            try:
                audio_list, video_list = [], []
                for vid_path, aud_path in batch_pairs:
                    audio_list.append(load_audio_fbank(aud_path, target_length))
                    video_list.append(load_video_frames(vid_path, TOTAL_FRAMES))

                # audio_batch : (n, target_length, 128)
                # video_batch : (n, TOTAL_FRAMES, 3, 224, 224)
                audio_batch = torch.stack(audio_list).to(DEVICE)
                video_batch = torch.stack(video_list).to(DEVICE)

                # CAV-MAE-Sync forward_feat expects:
                #   audio : (n * TOTAL_FRAMES, target_length, 128)
                #   video : (n * TOTAL_FRAMES, 3, 224, 224)
                # Each frame is paired with the full audio clip it belongs to.
                audio_rep = audio_batch.unsqueeze(1).expand(-1, TOTAL_FRAMES, -1, -1)
                audio_rep = audio_rep.reshape(n * TOTAL_FRAMES, target_length, 128)
                video_flat = video_batch.view(n * TOTAL_FRAMES, 3, 224, 224)

                model_core = model.module if hasattr(model, "module") else model
                result = model_core.forward_feat(audio_rep, video_flat)

                tokens_a, tokens_v = result[0], result[1]
                # tokens_a : (n * TOTAL_FRAMES, T_a, D)   T_a includes CLS + registers + patches
                # tokens_v : (n * TOTAL_FRAMES, T_v, D)

                # CLS token is at index 0; it is the clip-level summary
                cls_a = tokens_a[:, 0, :]   # (n * TOTAL_FRAMES, D)
                cls_v = tokens_v[:, 0, :]   # (n * TOTAL_FRAMES, D)

                # Average over the TOTAL_FRAMES dimension to get one vector per segment
                cls_a = cls_a.view(n, TOTAL_FRAMES, -1).mean(dim=1)  # (n, D)
                cls_v = cls_v.view(n, TOTAL_FRAMES, -1).mean(dim=1)  # (n, D)

                # L2-normalise each modality independently
                cls_a = F.normalize(cls_a, dim=-1)
                cls_v = F.normalize(cls_v, dim=-1)

                # Concatenate: final embedding is (n, D_a + D_v)
                combined = torch.cat([cls_a, cls_v], dim=-1)
                all_embeddings.append(combined.cpu().float().numpy())

                end = min(batch_start + BATCH_SIZE, len(paired))
                print(f"  Batch {batch_start // BATCH_SIZE + 1}: done ({end}/{len(paired)})")

            except Exception as exc:
                import traceback
                print(f"  ERROR in batch starting at {batch_start}: {type(exc).__name__}: {exc}")
                traceback.print_exc()
                continue

    if not all_embeddings:
        print("No embeddings extracted.")
        return None, None

    embeddings = np.concatenate(all_embeddings, axis=0).astype(np.float32)
    print(f"Extracted {embeddings.shape[0]} embeddings, shape: {embeddings.shape}")
    return embeddings, output_path


In [7]:
# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
all_results = {}

for seg_sec in SEGMENT_DURATIONS:
    embeddings, output_path = extract_embeddings_for_segment_duration(seg_sec)

    if embeddings is None:
        continue

    save_file = output_path / f"embeddings_{int(seg_sec)}s.npy"
    np.save(save_file, embeddings)
    print(f"Saved: {save_file}  shape={embeddings.shape}")
    all_results[seg_sec] = embeddings

print("\nDONE.")
print(f"Processed durations: {sorted(all_results.keys())}")
for seg_sec, emb in all_results.items():
    print(f"  {int(seg_sec)}s -> {emb.shape}")



Processing 2s segments
  Video 1: 106 paired chunks [OK: 106 vids / 106 auds]
  Video 2: 110 paired chunks [OK: 110 vids / 110 auds]
  Video 3: 93 paired chunks [OK: 93 vids / 93 auds]
  Video 4: 31 paired chunks [OK: 31 vids / 31 auds]
  Video 5: 41 paired chunks [OK: 41 vids / 41 auds]
  Video 6: 113 paired chunks [OK: 113 vids / 113 auds]
  Video 7: 129 paired chunks [OK: 129 vids / 129 auds]
  Video 8: 124 paired chunks [OK: 124 vids / 124 auds]
  Video 9: 41 paired chunks [OK: 41 vids / 41 auds]
  Video 10: 90 paired chunks [OK: 90 vids / 90 auds]
  Video 11: 92 paired chunks [OK: 92 vids / 92 auds]
  Video 12: 101 paired chunks [OK: 101 vids / 101 auds]
  Video 13: 70 paired chunks [OK: 70 vids / 70 auds]
  Video 14: 41 paired chunks [OK: 41 vids / 41 auds]
  Video 15: 116 paired chunks [OK: 116 vids / 116 auds]
  Video 16: 115 paired chunks [OK: 115 vids / 115 auds]
  Video 17: 127 paired chunks [OK: 127 vids / 127 auds]
  Video 18: 41 paired chunks [OK: 41 vids / 41 auds]
Foun